In [35]:
from dotenv import load_dotenv
load_dotenv()

True

## output parsing 
output parsing are classes that give model response in different formats
* there are two main methods an output parser implements
- ** get instruction format ** : This method return a string containing instruction for how the output
  of a language model should be formatted.
- ** Parser method ** : a method which takes string with instruction and outputs returns a different structre output (JSON,CSV,python,simple text) 

- Output Parsing types 
    - StrOutputParser
    - JsonOutputParser
    - CSVOutputParser
    - StructuredOutputParser(pydantic or json) genrate all type of format output

### 1. pydantic OutputParser 

In [36]:
from typing import Optional
from pydantic import BaseModel,Field
from langchain_core.output_parsers import PydanticOutputParser

In [37]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import (SystemMessagePromptTemplate,
                                    HumanMessagePromptTemplate,
                                    ChatPromptTemplate,PromptTemplate)
base_url ="http://localhost:11434"
model = 'llama3.1:8b'

llm = ChatOllama(base_url = base_url, model = model)


In [38]:
class ProductReview(BaseModel):
    """ product review given by a user """
    product_name: str = Field(description = "Name of the Product")
    review: str = Field(description = "User review text")
    rating: int = Field(description = "ratting from 1 to 5")
    verified_purchas: Optional[bool] = Field(description= "Weather the purchase is verified",default = None)

In [39]:
parser = PydanticOutputParser(pydantic_object=ProductReview)

# making format instruction 

In [40]:
instruction = parser.get_format_instructions()

In [41]:
print(instruction)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "product review given by a user ", "properties": {"product_name": {"description": "Name of the Product", "title": "Product Name", "type": "string"}, "review": {"description": "User review text", "title": "Review", "type": "string"}, "rating": {"description": "ratting from 1 to 5", "title": "Rating", "type": "integer"}, "verified_purchas": {"anyOf": [{"type": "boolean"}, {"type": "null"}], "default": null, "description": "Weather the purchase is verified", "title": "Verified Purchas"}}, "required": ["product_name", "review", "ra

In [42]:
prompt = PromptTemplate(
    template = '''
    write a product review based on the user query.
    follow the formatting instructions carefully.
    {format_instruction}

    User Query:{query}

    Product Review:
    ''',
    input_variables = ['query'],
    partial_variables = {'format_instruction': parser.get_format_instructions()}
) 
chain = prompt | llm 

In [43]:
output = chain.invoke({"query":"write a review for the bully girls "})

In [44]:
print(output.content)

{
  "product_name": "Bully Girls",
  "review": "I recently purchased the Bully Girls and I must say, it's been a game-changer. The quality is top-notch and the design is sleek and modern. The materials are durable and easy to clean, making it perfect for daily use. The best part? It's affordable without compromising on quality! 5/5 stars!",
  "rating": 5,
  "verified_purchas": true
}


Sturctured ouput parser 

In [45]:
structured_llm = llm.with_structured_output(ProductReview)

In [47]:
output = structured_llm.invoke("write a product review for apple air macbook m4 512gb")
print(output)

product_name='Apple MacBook Air M4 (2023) with 512GB Storage' review="5/5 stars. It's an extremely satisfying laptop that has not left me disappointed. The performance and battery life are excellent, making it perfect for both work and play. The sleek design is also something I appreciate. My only criticism is the lack of USB-A ports, but this shouldn't be a problem for most users. Overall, it's a fantastic device worth every penny." rating=5 verified_purchas=True


## Json Output Parser 
- Json output Parser accept a base message  as input and return json output

In [49]:
from langchain_core.output_parsers import JsonOutputParser

In [50]:
parser = JsonOutputParser(pydantic_object=ProductReview)
print(parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


create prompt template 

In [54]:
prompt = PromptTemplate(
    template= '''
    write a product review based on the user query 
    Follow the formatting instruction carefully.

    {format_instruction}

    User Query: {query}
    Product Review:
    ''', 
    input_variables= ['query'],
    partial_variables={'format_instruction': parser.get_format_instructions()}

)
chain = prompt | llm | parser
output = chain.invoke({'query': 'wite a review for lembo'})


In [55]:
print(output)

{'product_name': 'Lembo', 'review': "It's a nice product but the customer service was not good.", 'rating': 3, 'verified_purchase': None}


## CSV  Output Parser 
- This parser returns a list of comma seperated values 

In [58]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
parser = CommaSeparatedListOutputParser()
print(parser.get_format_instructions())

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [59]:
format_instruction = parser.get_format_instructions()

In [76]:
promt = PromptTemplate(
    template= '''
    write a product review based on the user query 
    Follow the formatting instruction carefully.

    {format_instruction}

    User Query: {query}
    Product Review:
    ''', 
    input_variables= ['query'],
    partial_variables={'format_instruction': format_instruction}
)

In [77]:
chain = promt | llm | parser
output = chain.invoke({'query':'Genrate csv keywords for Ai an machine learning'})
print(output)

['Here is a product review in the format you requested:', 'Artificial Intelligence', 'AI Technology', 'Machine Learning Models', 'Natural Language Processing', 'Deep Learning Techniques', 'Neural Networks', 'Cognitive Computing', 'Big Data Analytics', 'Predictive Maintenance', 'Data Science Tools', 'AutoML Libraries', 'Model Interpretability', 'TensorFlow', 'Keras', 'PyTorch', 'Scikit-Learn', 'Apache Spark', 'Hadoop', 'Spark MLlib', 'OpenCV']


### stroutput Parser task

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from typing import Optional

# ── 1. ProductReviews Model ───────────────────────────────────
class ProductReviews(BaseModel):
    """product review given by a user"""
    product_name:      str            = Field(description="Name of the Product")
    review:            str            = Field(description="User review text")
    rating:            int            = Field(description="Rating from 1 to 10")
    verified_purchase: Optional[bool] = Field(description="Whether the purchase is verified", default=None)


llm    = ChatOllama(model="llama3.1:8b")
parser = StrOutputParser()


prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert product reviewer.
When given a product name, generate a detailed review in this EXACT format:

Product Name : <name>
Review       : <detailed review>
Rating       : <number 1-10>
Verified     : <Yes or No>
"""),
    ("human", "Generate a product review for: {product_name}")
])


chain = prompt | llm | parser


print("=" * 55)
print("         PRODUCT REVIEW GENERATOR")
print("=" * 55)

response = chain.invoke({"product_name": "iPhone 15 Pro"})
print(response)


products = [
    "Samsung Galaxy S24",
    "Sony WH-1000XM5 Headphones",
    "MacBook Air M3"
]

print("\n" + "=" * 55)
print("       BATCH PRODUCT REVIEWS")
print("=" * 55)

for product in products:
    print(f"\n🛍️  Product: {product}")
    print("-" * 45)
    result = chain.invoke({"product_name": product})
    print(result)
    print()

# ── 7. Interactive Mode ───────────────────────────────────────
print("=" * 55)
print("          INTERACTIVE REVIEW MODE")
print("=" * 55)

while True:
    product = input("\n🔍 Enter product name (or 'quit' to exit): ")
    if product.lower() == 'quit':
        print("👋 Exiting...")
        break
    print("\n⏳ Generating review...\n")
    result = chain.invoke({"product_name": product})
    print(result)
    print("-" * 45)

task 2 

In [81]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
prompt = PromptTemplate(
    template= '''
    write a deatiled product review for the following product

    product: {product}

    Include:
    -features
    -pros
    -corns
    -final ratting from 1 to 5 
    ''',
    input_variables= ['product']
)
chain = prompt | llm | parser
output = chain.invoke({'product': 'macbook m3'})

In [82]:
print(output)

**Product Review: MacBook M3**

**Overview**

The MacBook M3 is Apple's latest addition to the MacBook lineup, powered by the M3 chip. As a long-time user of MacBooks, I was excited to get my hands on this new model and see how it stacks up against its predecessors.

**Features**

* **M3 Chip**: The M3 chip provides a significant boost in performance and efficiency compared to previous models.
* **8GB RAM**: 8GB of RAM is standard on the MacBook M3, which should be sufficient for most users' needs.
* **256GB Storage**: While some may find it limited, the base storage option is still plenty for most users who don't need to store a massive library of files and applications.
* **13.3-inch Retina Display**: The display on the MacBook M3 features a 13.3-inch Retina display with a resolution of 2560 x 1600 pixels.
* **Touch ID**: For added security, Touch ID is now available on the MacBook M3.
* **Thunderbolt 4**: Four Thunderbolt 4 ports offer fast data transfer and charging capabilities.

